# MGMT298D: Science and Strategy of AI
### Week 5A - Neural Networks
### Application: Fashion Image Classification

## Import Libraries and Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score

np.random.seed(42)
tf.random.set_seed(42)

CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

# Load Fashion-MNIST dataset
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Normalize and reshape
x_train_full = x_train_full.astype("float32").reshape(-1, 784) / 255.0
x_test = x_test.astype("float32").reshape(-1, 784) / 255.0

X_train, X_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
)

print(f"Training: {X_train.shape} | Validation: {X_val.shape} | Test: {x_test.shape}")

## Visualize Sample Images

In [ ]:
# Display pixel matrix for one example
print(f"Label: {y_train[6]} ({CLASS_NAMES[y_train[6]]})")
image_matrix = X_train[6].reshape(28, 28)
df_image = pd.DataFrame(image_matrix)
display(df_image.style.format("{:.2f}").hide())

plt.figure(figsize=(2, 2))
plt.imshow(X_train[6].reshape(28, 28), cmap="gray_r")
plt.title(f"Label: {CLASS_NAMES[y_train[6]]}")
plt.axis("off")
plt.show()

In [ ]:
# Show first 10 images
plt.figure(figsize=(18, 2.2))
for i in range(10):
    plt.subplot(1, 10, i+1)
    plt.imshow(X_train[i].reshape(28, 28), cmap="gray_r")
    plt.title(CLASS_NAMES[y_train[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

## Model A: XGBoost Baseline

In [ ]:
xgb_clf = xgb.XGBClassifier(random_state=42, n_estimators=10, max_depth=3)
xgb_clf.fit(x_train_full, y_train_full, verbose=True)

y_pred_xgb = xgb_clf.predict(x_test)
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)

print(f"\nModel A (XGBoost) — Test Accuracy: {xgb_accuracy*100:.2f}%")

## Model B: Simple Neural Network (1 Hidden Layer)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)

def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Validation')
    ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True)
    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Validation')
    ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True)
    fig.suptitle(title)
    plt.show()

model_B = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(10, activation="softmax")
])
model_B.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_B.summary()

history_B = model_B.fit(X_train, y_train, epochs=50, batch_size=128,
                        validation_data=(X_val, y_val), callbacks=[early_stopping], verbose=1)

test_loss_B, test_acc_B = model_B.evaluate(x_test, y_test, verbose=0)
print(f"Model B — Test Accuracy: {test_acc_B*100:.2f}%")
plot_history(history_B, "Model B: Simple NN")

## Model C: Deeper Neural Network (2 Hidden Layers)

In [ ]:
model_C = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(10, activation="softmax")
])
model_C.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_C.summary()

history_C = model_C.fit(X_train, y_train, epochs=50, batch_size=128,
                        validation_data=(X_val, y_val), callbacks=[early_stopping], verbose=1)

test_loss_C, test_acc_C = model_C.evaluate(x_test, y_test, verbose=0)
print(f"Model C — Test Accuracy: {test_acc_C*100:.2f}%")
plot_history(history_C, "Model C: Deeper NN")

## Model D: Neural Network with Dropout

In [ ]:
model_D = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(8, activation="relu"),
    layers.Dense(10, activation="softmax")
])
model_D.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_D.summary()

history_D = model_D.fit(X_train, y_train, epochs=50, batch_size=128,
                        validation_data=(X_val, y_val), callbacks=[early_stopping], verbose=1)

test_loss_D, test_acc_D = model_D.evaluate(x_test, y_test, verbose=0)
print(f"Model D — Test Accuracy: {test_acc_D*100:.2f}%")
plot_history(history_D, "Model D: NN with Dropout")

## Model E: Neural Network with Batch Normalization

In [ ]:
model_E = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(32, use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(16, use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(8, activation="relu"),
    layers.Dense(10, activation="softmax")
])
model_E.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_E.summary()

history_E = model_E.fit(X_train, y_train, epochs=50, batch_size=128,
                        validation_data=(X_val, y_val), callbacks=[early_stopping], verbose=1)

test_loss_E, test_acc_E = model_E.evaluate(x_test, y_test, verbose=0)
print(f"Model E — Test Accuracy: {test_acc_E*100:.2f}%")
plot_history(history_E, "Model E: NN with BatchNorm")

## Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['A (XGBoost)', 'B (1 Layer)', 'C (2 Layers)', 'D (Dropout)', 'E (BatchNorm)'],
    'Test Accuracy': [xgb_accuracy, test_acc_B, test_acc_C, test_acc_D, test_acc_E]
})
results['Test Accuracy'] = results['Test Accuracy'] * 100
results = results.sort_values(by='Test Accuracy', ascending=False).set_index('Model')
print(results.to_string(formatters={'Test Accuracy': '{:.2f}%'.format}))

## Classification Reports

In [ ]:
y_pred_B = np.argmax(model_B.predict(x_test), axis=1)
y_pred_C = np.argmax(model_C.predict(x_test), axis=1)
y_pred_D = np.argmax(model_D.predict(x_test), axis=1)
y_pred_E = np.argmax(model_E.predict(x_test), axis=1)

models = {
    'Model A (XGBoost)': y_pred_xgb,
    'Model B (1 Layer)': y_pred_B,
    'Model C (2 Layers)': y_pred_C,
    'Model D (Dropout)': y_pred_D,
    'Model E (BatchNorm)': y_pred_E
}

for name, y_pred in models.items():
    print(f"\n--- {name} ---")
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))